# 503 — Trace Context Smoke Test

Validates that authenticated user/session context propagates correctly from the app's
login boundary through to persisted trace metadata.

**No live Neo4j connection required.** An in-memory fake `TraceRepository` is used
wherever persistence would be called. All assertions fail loudly with a descriptive message.

## What is tested
1. `UserContext` built from an `AuthenticatedUser` shape has the expected fields
2. The investigation-start boundary produces the correct `UserContext` metadata dict
3. `Orchestrator.run()` accepts the new `user_context` kwarg and falls back gracefully on legacy `user_id`
4. `InvestigationTrace` carries all 5 identity fields
5. `TraceRepository.save_trace()` receives and stores all 5 identity fields
6. Older traces that lack the new fields still load gracefully
7. `TraceService.start_trace()` propagates metadata into the richer trace shape

## Manual checks still required
- End-to-end Streamlit flow: sign in → run investigation → load trace in Replay tab → verify Session Identity expander
- Neo4j property persistence: query `MATCH (t:InvestigationTrace) RETURN t.user_role, t.auth_provider, t.session_id, t.gateway_mode LIMIT 5`
- Jr-analyst regression: confirm investigation flow unchanged for restricted role

In [1]:
import sys
sys.path.insert(0, "..")
print("sys.path OK")

sys.path OK


## Test 1 — UserContext built from AuthenticatedUser shape

In [2]:
from src.app.auth import AuthenticatedUser
from src.domain.models import UserContext

# Simulate what authenticate() returns
auth_user = AuthenticatedUser(
    user_id="sr_analyst",
    role="sr_risk_analyst",
    auth_provider="mock",
)
assert auth_user.user_id == "sr_analyst", "user_id mismatch"
assert auth_user.role == "sr_risk_analyst", "role mismatch"
assert auth_user.auth_provider == "mock", "auth_provider mismatch"
assert len(auth_user.session_id) == 36, f"expected uuid, got {auth_user.session_id!r}"

# Simulate what layout.py builds in the _trigger_run handler
mcp_mode = "local"
ctx = UserContext(
    user_id=auth_user.user_id,
    session_id=auth_user.session_id,
    metadata={
        "role":          auth_user.role,
        "auth_provider": auth_user.auth_provider,
        "gateway_mode":  mcp_mode,
    },
)
assert ctx.user_id == "sr_analyst", "ctx.user_id"
assert ctx.session_id == auth_user.session_id, "ctx.session_id"
assert ctx.metadata["role"] == "sr_risk_analyst", "metadata.role"
assert ctx.metadata["auth_provider"] == "mock", "metadata.auth_provider"
assert ctx.metadata["gateway_mode"] == "local", "metadata.gateway_mode"

print("Test 1 PASSED — UserContext built from AuthenticatedUser shape")

Test 1 PASSED — UserContext built from AuthenticatedUser shape


## Test 2 — Investigation start boundary constructs correct UserContext

In [3]:
# Verify the UserContext from Test 1 carries all required fields for the trace
required_keys = {"role", "auth_provider", "gateway_mode"}
missing = required_keys - ctx.metadata.keys()
assert not missing, f"Missing metadata keys: {missing}"

assert ctx.user_id, "user_id must be non-empty"
assert ctx.session_id, "session_id must be non-empty"
assert ctx.metadata["role"], "role must be non-empty"
assert ctx.metadata["auth_provider"], "auth_provider must be non-empty"
assert ctx.metadata["gateway_mode"] in ("local", "remote"), (
    f"unexpected gateway_mode: {ctx.metadata['gateway_mode']!r}"
)

print("Test 2 PASSED — investigation start boundary UserContext is correct")

Test 2 PASSED — investigation start boundary UserContext is correct


## Test 3 — `InvestigationTrace` stores all 5 identity fields

In [4]:
from src.domain.models import InvestigationTrace

trace = InvestigationTrace(
    request_id="test-run-id-001",
    entity_name="ACME LTD",
    question="Who owns ACME LTD?",
    user_id=ctx.user_id,
    mode="investigate",
    user_role=ctx.metadata.get("role", ""),
    auth_provider=ctx.metadata.get("auth_provider", ""),
    session_id=ctx.session_id,
    gateway_mode=ctx.metadata.get("gateway_mode", ""),
)

assert trace.user_id == "sr_analyst",         f"user_id: {trace.user_id!r}"
assert trace.user_role == "sr_risk_analyst",   f"user_role: {trace.user_role!r}"
assert trace.auth_provider == "mock",          f"auth_provider: {trace.auth_provider!r}"
assert trace.session_id == ctx.session_id,     f"session_id: {trace.session_id!r}"
assert trace.gateway_mode == "local",          f"gateway_mode: {trace.gateway_mode!r}"

print("Test 3 PASSED — InvestigationTrace carries all 5 identity fields")

Test 3 PASSED — InvestigationTrace carries all 5 identity fields


## Test 4 — `TraceRepository.save_trace()` receives all 5 identity fields

Uses an in-memory fake repository that records the Cypher parameters passed to it.

In [5]:
from src.storage.trace_repository import TraceRepository

# In-memory fake Neo4jRepository — records run_query calls, returns empty lists
class FakeNeo4jRepo:
    def __init__(self):
        self.calls = []  # list of (cypher, params) tuples

    def run_query(self, cypher, params):
        self.calls.append((cypher, dict(params)))
        return []

fake_neo4j = FakeNeo4jRepo()
repo = TraceRepository(fake_neo4j)

repo.save_trace(trace)

assert fake_neo4j.calls, "save_trace() should have called run_query at least once"
saved_params = fake_neo4j.calls[0][1]

assert saved_params.get("user_id") == "sr_analyst",        f"user_id: {saved_params.get('user_id')!r}"
assert saved_params.get("user_role") == "sr_risk_analyst",  f"user_role: {saved_params.get('user_role')!r}"
assert saved_params.get("auth_provider") == "mock",         f"auth_provider: {saved_params.get('auth_provider')!r}"
assert saved_params.get("session_id") == ctx.session_id,    f"session_id: {saved_params.get('session_id')!r}"
assert saved_params.get("gateway_mode") == "local",         f"gateway_mode: {saved_params.get('gateway_mode')!r}"

print("Test 4 PASSED — save_trace() receives all 5 identity fields")

Test 4 PASSED — save_trace() receives all 5 identity fields


## Test 5 — `Orchestrator.run()` accepts `user_context` and uses it in the trace

Uses a minimal stub to avoid requiring a live database or AI client.

In [6]:
from unittest.mock import MagicMock, patch
from src.orchestration.orchestrator import Orchestrator
from src.orchestration.planner import PlannerResult

# Stub planner that returns a minimal trace-mode plan (no entities to resolve)
def _stub_plan(query, allowed_tools=None):
    return PlannerResult(
        mode="trace",
        reason="smoke test",
        entities=[],
        plan=[],
        stop_conditions=[],
        raw={},
    )

captured_traces = []

class CapturingFakeNeo4j:
    def run_query(self, cypher, params):
        if "InvestigationTrace" in cypher:
            captured_traces.append(dict(params))
        return []

fake_repo_inner = CapturingFakeNeo4j()
from src.storage.trace_repository import TraceRepository as TR
from src.tracing.trace_service import TraceService as TS

trace_repo_stub = TR(fake_repo_inner)
trace_svc_stub = TS(trace_repo_stub)

orchestrator = Orchestrator(
    planner=MagicMock(plan=_stub_plan),
    mcp=MagicMock(call_tool=lambda *a, **kw: MagicMock(success=True, data={}, error=None)),
    graph_agent=MagicMock(),
    risk_agent=MagicMock(),
    trace_agent=MagicMock(),
    trace_service=trace_svc_stub,
    trace_repo=trace_repo_stub,
    ai_client=None,
)

# Run with the new user_context kwarg
result = orchestrator.run("smoke test query", user_context=ctx)

assert result.success is not None, "run() must return an OrchestratorResult"
assert captured_traces, "save_trace should have been called"
p = captured_traces[0]
assert p.get("user_id") == "sr_analyst",       f"user_id in trace: {p.get('user_id')!r}"
assert p.get("user_role") == "sr_risk_analyst", f"user_role in trace: {p.get('user_role')!r}"
assert p.get("auth_provider") == "mock",        f"auth_provider in trace: {p.get('auth_provider')!r}"
assert p.get("session_id"),                     "session_id must be non-empty"
assert p.get("gateway_mode") == "local",        f"gateway_mode in trace: {p.get('gateway_mode')!r}"

print("Test 5 PASSED — Orchestrator.run(user_context=...) threads identity into trace")

[04/02/26 16:51:39] INFO     [89dd43ad] Investigation started: 'smoke test query'               ]8;id=989795;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py\orchestrator.py]8;;\:]8;id=185018;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py#433\433]8;;\

                    INFO     [89dd43ad] Routing: mode=trace entities=[] steps=0                 ]8;id=849894;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py\orchestrator.py]8;;\:]8;id=568529;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py#453\453]8;;\

                    INFO     [89dd43ad] Investigation complete: success=False mode=trace        ]8;id=797789;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py\orchestrator.py]8;;\:]8;id=629999;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py#836\836]8;;\
                             steps=0/0 skipped=0                                                                   

Test 5 PASSED — Orchestrator.run(user_context=...) threads identity into trace


## Test 6 — Legacy `user_id` path still works (backward compatibility)

In [7]:
legacy_captures = []

class LegacyCapturingNeo4j:
    def run_query(self, cypher, params):
        if "InvestigationTrace" in cypher:
            legacy_captures.append(dict(params))
        return []

leg_inner = LegacyCapturingNeo4j()
leg_repo = TR(leg_inner)
leg_svc = TS(leg_repo)

legacy_orch = Orchestrator(
    planner=MagicMock(plan=_stub_plan),
    mcp=MagicMock(call_tool=lambda *a, **kw: MagicMock(success=True, data={}, error=None)),
    graph_agent=MagicMock(),
    risk_agent=MagicMock(),
    trace_agent=MagicMock(),
    trace_service=leg_svc,
    trace_repo=leg_repo,
    ai_client=None,
)

# Call with old-style user_id — no user_context supplied
result_legacy = legacy_orch.run("legacy query", user_id="notebook_caller")

assert result_legacy is not None, "legacy run() must return a result"
assert legacy_captures, "save_trace should be called for legacy path"
lp = legacy_captures[0]
assert lp.get("user_id") == "notebook_caller", f"legacy user_id: {lp.get('user_id')!r}"
assert lp.get("user_role", "") == "",           "legacy user_role should be empty string"
assert lp.get("auth_provider", "") == "",       "legacy auth_provider should be empty string"
assert lp.get("session_id", "") == "",          "legacy session_id should be empty string"

print("Test 6 PASSED — legacy user_id path works unchanged")

                    INFO     [8df2d6e8] Investigation started: 'legacy query'                   ]8;id=560728;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py\orchestrator.py]8;;\:]8;id=782790;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py#433\433]8;;\

                    INFO     [8df2d6e8] Routing: mode=trace entities=[] steps=0                 ]8;id=267002;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py\orchestrator.py]8;;\:]8;id=501672;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py#453\453]8;;\

                    INFO     [8df2d6e8] Investigation complete: success=False mode=trace        ]8;id=342624;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py\orchestrator.py]8;;\:]8;id=827623;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/orchestration/orchestrator.py#836\836]8;;\
                             steps=0/0 skipped=0                                                                   

Test 6 PASSED — legacy user_id path works unchanged


## Test 7 — Older traces without new fields load gracefully

In [8]:
# Simulate what load_trace() returns for a pre-Prompt-3 trace node
# (Neo4j returns None for absent properties; Python dict.get() handles it)
old_trace_dict = {
    "trace_id":   "old-trace-001",
    "query":      "SOME CO",
    "question":   "Who owns SOME CO?",
    "user_id":    "jr_analyst",
    "mode":       "investigate",
    "started_at": "2025-01-15T10:00:00+00:00",
    "ended_at":   "2025-01-15T10:02:30+00:00",
    "final_summary": "Investigation complete.",
    # NOTE: user_role, auth_provider, session_id, gateway_mode are absent
    #       (old traces stored before Prompt 3 was deployed)
    "events": [],
}

# The UI uses .get() with fallback — verify it handles None/absent keys
assert old_trace_dict.get("user_role", "") == "",      "missing user_role should default to empty"
assert old_trace_dict.get("auth_provider", "") == "",  "missing auth_provider should default to empty"
assert old_trace_dict.get("session_id", "") == "",     "missing session_id should default to empty"
assert old_trace_dict.get("gateway_mode", "") == "",   "missing gateway_mode should default to empty"

# Verify None values (as returned by the Neo4j driver) are also safe
none_trace = {**old_trace_dict, "user_role": None, "auth_provider": None}
assert (none_trace.get("user_role") or "") == "",      "None user_role should coerce to empty"
assert (none_trace.get("auth_provider") or "") == "",  "None auth_provider should coerce to empty"

print("Test 7 PASSED — older traces load gracefully without new fields")

Test 7 PASSED — older traces load gracefully without new fields


## Test 8 — `TraceService.start_trace()` propagates metadata into the trace

In [9]:
from src.domain.models import InvestigationRequest, UserContext
from src.tracing.trace_service import TraceService

svc_captures = []

class SvcCapturingNeo4j:
    def run_query(self, cypher, params):
        if "InvestigationTrace" in cypher:
            svc_captures.append(dict(params))
        return []

from src.storage.trace_repository import TraceRepository as TR2
svc_repo = TR2(SvcCapturingNeo4j())
svc = TraceService(svc_repo)

rich_ctx = UserContext(
    user_id="jr_analyst",
    session_id="abc-session-123",
    metadata={
        "role":          "jr_risk_analyst",
        "auth_provider": "mock",
        "gateway_mode":  "remote",
        "mode":          "investigate",
    },
)
req = InvestigationRequest(
    entity_name="TEST CO",
    context=rich_ctx,
    request_id="req-smoke-001",
)

t = svc.start_trace(req, rich_ctx)

assert t.user_id == "jr_analyst",        f"t.user_id: {t.user_id!r}"
assert t.user_role == "jr_risk_analyst", f"t.user_role: {t.user_role!r}"
assert t.auth_provider == "mock",        f"t.auth_provider: {t.auth_provider!r}"
assert t.session_id == "abc-session-123",f"t.session_id: {t.session_id!r}"
assert t.gateway_mode == "remote",       f"t.gateway_mode: {t.gateway_mode!r}"

assert svc_captures, "start_trace should have called save_trace"
sp = svc_captures[0]
assert sp["user_role"] == "jr_risk_analyst", f"saved user_role: {sp['user_role']!r}"
assert sp["auth_provider"] == "mock",         f"saved auth_provider: {sp['auth_provider']!r}"
assert sp["gateway_mode"] == "remote",        f"saved gateway_mode: {sp['gateway_mode']!r}"

print("Test 8 PASSED — TraceService.start_trace() propagates metadata into the trace")

Test 8 PASSED — TraceService.start_trace() propagates metadata into the trace


## Test 9 — Compile check

In [10]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "compileall", "-q", "../src", "../app.py"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise AssertionError("compileall failed — syntax error in changed files")
print("Test 9 PASSED — all source files compile cleanly")

Test 9 PASSED — all source files compile cleanly


## Summary

All 9 tests pass if no `AssertionError` was raised above.

| # | Test | Coverage |
|---|------|----------|
| 1 | `UserContext` from `AuthenticatedUser` shape | Domain model construction |
| 2 | Investigation start boundary metadata | layout.py trigger handler |
| 3 | `InvestigationTrace` carries 5 identity fields | models.py |
| 4 | `save_trace()` receives all 5 fields | trace_repository.py |
| 5 | `Orchestrator.run(user_context=...)` end-to-end | orchestrator.py |
| 6 | Legacy `user_id` path unchanged | backward compatibility |
| 7 | Old traces without new fields load gracefully | backward compatibility |
| 8 | `TraceService.start_trace()` propagates metadata | trace_service.py |
| 9 | `compileall` passes | syntax validation |

### Manual checks still required
- Streamlit flow: sign in → run investigation → load trace ID in Replay/Audit tab → confirm 🔑 Session Identity expander appears with all 5 fields
- Neo4j confirmation: `MATCH (t:InvestigationTrace) RETURN t.user_role, t.auth_provider, t.session_id, t.gateway_mode LIMIT 5`
- Jr-analyst regression: run an investigation as `jr_analyst`; verify no errors and that denied tools are skipped as before